# Export all DE comparisons (both projects) → harmonized CSV + Excel

One cell, run top to bottom. Reads every DE comparison from **A04 (organoid/butyrate, pydeseq2)** and
**A06 (HT29 co-infection, limma-voom)**, maps DESeq2 *and* limma column names to one schema, writes:
- `results/DE_export_all/DE_all_comparisons.xlsx` (one sheet per comparison + INDEX)
- `results/DE_export_all/csv/*.csv` (one per comparison) and a zip

Harmonized columns: `project, comparison, gene_id, gene_symbol, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj`.

**Fix vs earlier:** A06 tables use limma names (`logFC, AveExpr, P.Value, adj.P.Val`) — now mapped
correctly, so pvalue/padj are populated. A hard assertion stops with the real column names if anything
can't be mapped (no more silent blanks). Kernel: any pandas env with openpyxl.

In [ ]:
import os, re, glob, sys, zipfile
from pathlib import Path
from collections import defaultdict
import pandas as pd, numpy as np

# ---------- locate projects (handles Projects_shared and /storage/.../Projects) ----------
def first_existing(cands, label):
    for p in cands:
        if p and Path(p).exists():
            return Path(p)
    sys.exit(f"ERROR: could not find {label}. Tried:\n  " + "\n  ".join(str(c) for c in cands))

A04 = first_existing([os.environ.get('A04_PROJ'),
    Path.home()/'Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate',
    '/storage/users/job37yv/Projects/DECIDE_A04_CdiffOrganoid_Butyrate'], 'A04 project')
A06 = first_existing([os.environ.get('A06_PROJ'),
    Path.home()/'Projects_shared/epithelial_coinfection-model_WORKING',
    '/storage/users/job37yv/Projects/epithelial_coinfection-model_WORKING'], 'A06 project')

OUT = A04/'results/DE_export_all'; OUT.mkdir(parents=True, exist_ok=True)
CSVDIR = OUT/'csv'; CSVDIR.mkdir(exist_ok=True)
print(f"A04 = {A04}\nA06 = {A06}\nOUT = {OUT}\n")

# ---------- column mapping: DESeq2 + limma/edgeR names ----------
def norm(s): return re.sub(r'[^a-z0-9]', '', str(s).lower())
SYN = {
 'gene_id'        : ['geneid','ensembl','ensemblgeneid','ensemblid','id','feature','unnamed0','index'],
 'gene_symbol'    : ['symbol','genesymbol','genename','externalgenename','hgncsymbol'],
 'baseMean'       : ['basemean','aveexpr','meanexpr','avgexpr','baseexpr'],       # AveExpr (limma)
 'log2FoldChange' : ['log2foldchange','logfc','log2fc','lfc','l2fc'],             # logFC (limma)
 'lfcSE'          : ['lfcse','se','stderr'],
 'stat'           : ['stat','statistic','t','tvalue','wald','zstat'],
 'pvalue'         : ['pvalue','pval','pvalues'],                                  # P.Value -> 'pvalue'
 'padj'           : ['padj','adjpval','adjpvalue','adjustedpvalue','fdr','qvalue'],# adj.P.Val -> 'adjpval'
}
def build_mapping(cols):
    ncols = {c: norm(c) for c in cols}; used=set(); mp={}
    for target, cands in SYN.items():
        found=None
        for cand in cands:
            for orig, nc in ncols.items():
                if orig in used: continue
                if nc == cand: found=orig; break
            if found: break
        if not found:  # loose contains fallback
            for cand in cands:
                for orig, nc in ncols.items():
                    if orig in used: continue
                    if cand in nc: found=orig; break
                if found: break
        if found: mp[target]=found; used.add(found)
    return mp

def completeness(path):
    try: d = pd.read_csv(path, nrows=5)
    except Exception: return -1
    mp = build_mapping(d.columns)
    return sum(k in mp for k in ['log2FoldChange','pvalue','padj','baseMean'])

def standardise(path, project, comparison):
    d = pd.read_csv(path)
    mp = build_mapping(d.columns)
    missing = [k for k in ['log2FoldChange','pvalue','padj'] if k not in mp]
    if missing:
        raise ValueError(f"{os.path.basename(path)}: could not map {missing}. "
                         f"Actual columns = {list(d.columns)}")
    out = pd.DataFrame()
    out['gene_id']     = (d[mp['gene_id']] if 'gene_id' in mp else d.index).astype(str).str.replace(r'\..*','',regex=True)
    out['gene_symbol'] = d[mp['gene_symbol']].astype(str) if 'gene_symbol' in mp else out['gene_id']
    for k in ['baseMean','log2FoldChange','lfcSE','stat','pvalue','padj']:
        out[k] = pd.to_numeric(d[mp[k]], errors='coerce') if k in mp else np.nan
    out.insert(0,'project',project); out.insert(1,'comparison',comparison)
    return out

# ---------- discover files ----------
a04_files = sorted(glob.glob(str(A04/'results/tables/DE_*.csv')))
a06_glob  = sorted(set(glob.glob(str(A06/'analysis/DE/pydeseq2_*_vs_*.csv')) +
                       glob.glob(str(A06/'analysis/DE/python_deseq2_*_vs_*.csv'))))
def contrast_key(fn):
    m = re.search(r'(?:pydeseq2|python_deseq2)_([A-Za-z0-9]+_vs_[A-Za-z0-9]+)', os.path.basename(fn))
    return m.group(1) if m else os.path.basename(fn)
a06_by = defaultdict(list)
for f in a06_glob: a06_by[contrast_key(f)].append(f)

if not a04_files: print("WARNING: no A04 DE_*.csv found")
if not a06_by:    print("WARNING: no A06 pydeseq2/python_deseq2 files found")

def a04_name(f): return f"A04:{re.sub(r'^DE_|\.csv$','',os.path.basename(f))}"
def safe(s): return re.sub(r'[^A-Za-z0-9]+','_',s).strip('_')

tables={}; index_rows=[]
# A04
for f in a04_files:
    name=a04_name(f); df=standardise(f,'A04',name); tables[name]=df
    index_rows.append(dict(sheet=name, project='A04 organoid/butyrate', source=os.path.basename(f),
                           n_genes=len(df), n_sig=int((df.padj<0.05).sum()),
                           nn_pvalue=int(df.pvalue.notna().sum()), nn_padj=int(df.padj.notna().sum())))
# A06 — prefer a fully-mapping variant, and among those prefer the annotated (has Symbol)
for contrast, variants in a06_by.items():
    scored = sorted(variants, key=lambda f:(completeness(f), 1 if 'annotated' in os.path.basename(f) else 0))
    best = scored[-1]
    name=f"A06:{contrast}"; df=standardise(best,'A06',name); tables[name]=df
    index_rows.append(dict(sheet=name, project='A06 HT29 co-infection', source=os.path.basename(best),
                           n_genes=len(df), n_sig=int((df.padj<0.05).sum()),
                           nn_pvalue=int(df.pvalue.notna().sum()), nn_padj=int(df.padj.notna().sum())))

index = pd.DataFrame(index_rows)
print("Comparisons exported:")
print(index.to_string(index=False)); print()

# ---------- write CSVs + zip ----------
for name, df in tables.items():
    df.to_csv(CSVDIR/f'{safe(name)}.csv', index=False)
zpath = OUT/'DE_all_comparisons_csv.zip'
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
    for p in CSVDIR.glob('*.csv'): z.write(p, p.name)

# ---------- write Excel ----------
xlsx = OUT/'DE_all_comparisons.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
    index.to_excel(xw, sheet_name='INDEX', index=False)
    for name, df in tables.items():
        df.to_excel(xw, sheet_name=safe(name)[:31], index=False)

# ---------- final assertion: prove nothing is blank ----------
bad = index[(index.nn_pvalue==0) | (index.nn_padj==0)]
print(f"Wrote {len(tables)} comparisons")
print(f"  Excel : {xlsx}")
print(f"  CSVs  : {CSVDIR}")
print(f"  Zip   : {zpath}")
if len(bad):
    print("\n!!! WARNING: these still have empty pvalue/padj — check source columns:")
    print(bad.to_string(index=False))
else:
    print("\nOK — every comparison has populated pvalue and padj. ✅")


## Check after running
- The printed **INDEX** table should show non-zero `nn_pvalue` and `nn_padj` for **every** row.
- The last line should read **“OK — every comparison has populated pvalue and padj. ✅”**.
- If any row is empty, it prints that file's real columns so the synonym can be added in one line.

Note for the FAIR step: A04 = pydeseq2, A06 = limma-voom — this export harmonizes both to one schema.